# Lesson 3: Read the board

Every Symphony repository has a board: one page, STATE.md on the branch `board`, that says who holds what, what is free, and what must come first. A workflow rewrites it every time an issue, a branch or a pull request changes, so nobody edits it by hand and it is never stale. In this lesson you play a teaching assistant who keeps a course's repository, write down its state, and read the board the real code makes from it.

## Step 0: open a practice folder

Run this cell first. It makes an empty practice folder, `$LAB`, and fetches the claim-protocol library; every later block starts with `cd "$LAB"`, so nothing touches your real projects.

In [ ]:
# The practice folder, and the newest release of the public claim-protocol library.
import os, re, subprocess, tempfile
tags = subprocess.run(["git", "ls-remote", "--tags", "https://github.com/OpenSpineConsortium/symphony-protocol"], capture_output=True, text=True, check=True).stdout
releases = sorted({tuple(map(int, m)) for m in re.findall(r"refs/tags/v(\d+)\.(\d+)\.(\d+)$", tags, re.M)})
TAG = "v%d.%d.%d" % releases[-1]
LAB = tempfile.mkdtemp(prefix="symphony-lesson-")
LIBRARY = os.path.join(tempfile.mkdtemp(), "symphony-protocol")
subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--quiet", "--depth", "1",
                "--branch", TAG, "https://github.com/OpenSpineConsortium/symphony-protocol", LIBRARY], check=True)
os.environ.update({"GIT_CONFIG_GLOBAL": "/dev/null", "GIT_CONFIG_NOSYSTEM": "1", "GIT_AUTHOR_DATE": "2026-10-20T09:00:00Z", "GIT_COMMITTER_DATE": "2026-10-20T09:00:00Z", "GIT_CONFIG_COUNT": "4", "GIT_CONFIG_KEY_0": "gc.auto", "GIT_CONFIG_VALUE_0": "0", "GIT_CONFIG_KEY_1": "gc.autoDetach", "GIT_CONFIG_VALUE_1": "false", "GIT_CONFIG_KEY_2": "maintenance.auto", "GIT_CONFIG_VALUE_2": "false", "GIT_CONFIG_KEY_3": "maintenance.autoDetach", "GIT_CONFIG_VALUE_3": "false"})
os.environ.update(LAB=LAB, PYTHONPATH=LIBRARY)
print("symphony-protocol", TAG, "; practice folder:", LAB)

## Step 1: describe the course's repository

The board workflow asks GitHub's API for the open issues, the claim branches, the claim files and the pull requests. Here you write the same facts by hand into `course.json`, the shape the board accepts for a dry run. The course has a claimed task, a task in the queue, and nothing else yet.

In [ ]:
%%bash
cd "$LAB"
cat > course.json <<'JSON'
{
  "sha": "0123abcdef0123abcdef0123abcdef0123abcdef",
  "default_branch": "main",
  "issues": [
    {"number": 7, "title": "Fix the dosage table", "labels": [{"name": "claimed"}, {"name": "worker:you"}]},
    {"number": 8, "title": "Proofread chapter 2", "labels": [{"name": "todo"}]}
  ],
  "pulls": [],
  "branches": ["main", "claude/7"],
  "claims": {"7": "- Worker: you\n\n## Files to touch\n\n- chapters/dosage.md\n"},
  "merged": []
}
JSON
echo "written"

You should see lines like these:

```
written
```

## Step 2: render the board

This is the program the board workflow runs, told to read your file instead of GitHub.

In [ ]:
%%bash
cd "$LAB"
python3 -m sdlc_protocol board --dry-run course.json > STATE.md
grep -A5 "^## Claims" STATE.md
grep -A5 "^## Queue" STATE.md

You should see lines like these:

```
## Claims
| #7 | claude/7 | you |
chapters/dosage.md
## Queue
| #8 | Proofread chapter 2 | todo |
```

Read it like a student looking for work: #7 is taken (you hold it, and the board shows the file you said you would touch), and #8 is free in the Queue. So Sam, from lessons 1 and 2, takes #8.

## Step 3: give a task to one person

The teacher wants Maya, not just anyone, to write the summary. A label `worker:maya` on the issue gives it to her: it leaves the Queue and appears under Given to workers until she claims it.

In [ ]:
%%bash
cd "$LAB"
python3 - <<'PY'
import json
course = json.load(open("course.json"))
course["issues"].append({"number": 9, "title": "Write the chapter summary", "labels": [{"name": "todo"}, {"name": "worker:maya"}]})
json.dump(course, open("course.json", "w"))
PY
python3 -m sdlc_protocol board --dry-run course.json > STATE.md
grep -A5 "^## Given to workers" STATE.md
grep -c "Write the chapter summary" STATE.md

You should see lines like these:

```
## Given to workers
| #9 | Write the chapter summary | maya |
1
```

The summary appears once: under Given to workers, not in the Queue, so nobody else picks it up.

## Step 4: something breaks for everyone

The course's build stops working. An issue labelled `blocker` stops everyone: it is listed under Blockers, and a worker takes a blocker before anything in the Queue ([PROTOCOL.md](../../agents/PROTOCOL.md), Blockers).

In [ ]:
%%bash
cd "$LAB"
python3 - <<'PY'
import json
course = json.load(open("course.json"))
course["issues"].append({"number": 10, "title": "The PDF build is broken", "labels": [{"name": "blocker"}]})
json.dump(course, open("course.json", "w"))
PY
python3 -m sdlc_protocol board --dry-run course.json > STATE.md
grep -A5 "^## Blockers" STATE.md

You should see lines like these:

```
## Blockers
| #10 | The PDF build is broken | blocker |
```

## What you learned

- The board is made by code from what GitHub knows, so it is always current; you read it, you never edit it.
- Where to look for work, in order: Given to workers (yours first), then Blockers, then the Queue.
- In a real repository you read it with `/board` in Claude Code, or `git fetch origin +refs/heads/board:refs/remotes/origin/board && git show origin/board:STATE.md`.
- Next: [lesson 4](04-overlaps.ipynb), when two people's work touches the same file.